In [4]:
# Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import files

In [ ]:
# Upload both raw dataset files
uploaded = files.upload()

In [3]:
# Load csv data
fake = pd.read_csv("Fake_raw.csv")
true = pd.read_csv("True_raw.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset/Fake_raw.csv'

In [ ]:
# Drop subject and date columns (because they have no value for training)
for df in (fake, true):
    for col in ["subject", "date"]:
        if col in df.columns:
            df.drop(columns=col, inplace=True, errors="ignore")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Parameter size (FP32): 416.1 MB


In [ ]:
# Clean title columns
def clean_title(x):
    s = str(x)
    s = s.strip()  # trim surrounding whitespace/newlines

    # remove surrounding quote characters
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]

    # remove leading blanks/tabs
    s = s.lstrip(" \t")
    return s

for df in (fake, true):
    if "title" in df.columns:
        df["title"] = df["title"].apply(clean_title)

In [ ]:
# Add status column (fake = 0 and true = 1)
fake["status"] = 0
true["status"] = 1

In [ ]:
# Combine both files & shuffle
data = pd.concat([fake, true], ignore_index=True)
data = data.sample(frac=1.0, random_state=42).reset_index(drop=True)

In [ ]:
# Split 60% train, 20% val and 20% test (stratified by status)
train_df, temp_df = train_test_split(
    data, test_size=0.4, stratify=data["status"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["status"], random_state=42
)

In [ ]:
# Save to files
train_df.to_csv("train.csv", index=False)
val_df.to_csv("eval.csv", index=False)
test_df.to_csv("test.csv", index=False)